# cv2.imshow() is disabled in Colab
# Hence result cannot be displayed

In [2]:
# 1. Perform basic color-based segmentation to separate the blue color in an image

import cv2
import numpy as np

# Load the image
image = cv2.imread('/content/sample_data/crowd.jpeg')

# Convert to HSV color space
hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

# Define the blue color range
lower_blue = np.array([100, 150, 0])
upper_blue = np.array([140, 255, 255])

# Create a mask for the blue color
blue_mask = cv2.inRange(hsv_image, lower_blue, upper_blue)

# Apply the mask to the image
blue_segmented_image = cv2.bitwise_and(image, image, mask=blue_mask)




In [4]:
# 2. Use edge detection with Canny to highlight object edges in an image

import cv2

# Load the image
image = cv2.imread('/content/sample_data/crowd.jpeg', cv2.IMREAD_GRAYSCALE)

# Apply Canny edge detection
edges = cv2.Canny(image, 100, 200)




In [5]:
# 3. Load a pretrained Mask R-CNN model from PyTorch and use it for object detection and segmentation on an image

import torch
import torchvision
import cv2
import numpy as np

# Load a pretrained Mask R-CNN model
model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
model.eval()

# Load the image
image = cv2.imread('/content/sample_data/crowd.jpeg')
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image_tensor = torch.from_numpy(image_rgb / 255.0).permute(2, 0, 1).float().unsqueeze(0)

# Perform inference
with torch.no_grad():
    predictions = model(image_tensor)

# Extract the masks and bounding boxes
masks = (predictions[0]['masks'] > 0.5).squeeze().detach().cpu().numpy()
boxes = predictions[0]['boxes'].detach().cpu().numpy()



/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth
100%|██████████| 170M/170M [00:01<00:00, 140MB/s]


In [ ]:
# 4. Generate bounding boxes for each object detected by Mask R-CNN in an image


for box in boxes:
    cv2.rectangle(image, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (255, 0, 0), 2)  # Bounding box in blue

cv2.imshow('Bounding Boxes', image)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [7]:
# 5. Convert an image to grayscale and apply Otsu's thresholding method for segmentation

import cv2

# Load the image
image = cv2.imread('/content/sample_data/crowd.jpeg')

# Convert to grayscale
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Apply Otsu's thresholding
_, otsu_thresh = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)




In [9]:
# 6. Perform contour detection in an image to detect distinct objects or shapes

import cv2

# Load the image
image = cv2.imread('/content/sample_data/crowd.jpeg')
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray_image, 150, 255, cv2.THRESH_BINARY)

# Find contours
contours, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

# Draw contours on the original image
cv2.drawContours(image, contours, -1, (0, 255, 0), 3)



array([[[  0, 255,   0],
        [  0, 255,   0],
        [  0, 255,   0],
        ...,
        [ 87, 111, 165],
        [ 86, 108, 166],
        [ 80, 104, 162]],

       [[  0, 255,   0],
        [  0, 255,   0],
        [  0, 255,   0],
        ...,
        [ 91, 114, 170],
        [ 92, 116, 174],
        [ 95, 119, 177]],

       [[  0, 255,   0],
        [  0, 255,   0],
        [  0, 255,   0],
        ...,
        [ 86, 108, 166],
        [ 83, 107, 165],
        [ 87, 111, 171]],

       ...,

       [[  0, 255,   0],
        [  0, 255,   0],
        [  0, 255,   0],
        ...,
        [ 28,  22,  15],
        [ 28,  22,  15],
        [ 26,  20,  13]],

       [[  0, 255,   0],
        [  0, 255,   0],
        [  0, 255,   0],
        ...,
        [ 26,  20,  13],
        [ 28,  20,  13],
        [ 25,  19,  12]],

       [[  0, 255,   0],
        [  0, 255,   0],
        [  0, 255,   0],
        ...,
        [ 24,  16,   9],
        [ 28,  18,  11],
        [ 27,  19,  12]]

In [10]:
# 8. Apply k-means clustering for segmenting regions in an image

import cv2
import numpy as np

# Load the image
image = cv2.imread('/content/sample_data/crowd.jpeg')
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
pixel_values = image.reshape((-1, 3))
pixel_values = np.float32(pixel_values)

# Define criteria, number of clusters (K) and apply k-means
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
K = 3
_, labels, centers = cv2.kmeans(pixel_values, K, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

# Convert back to 8 bit values
centers = np.uint8(centers)

# Flatten the labels array
labels = labels.flatten()

# Convert all pixels to the color of the centroids
segmented_image = centers[labels.flatten()]

# Reshape back to the original image dimension
segmented_image = segmented_image.reshape(image.shape)


